# bridge

> IPython namespace bridge

In [ ]:
#| default_exp bridge

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
import paar.bridge as B
from paar.bridge import Bridge, on_change

class _Events:
    def __init__(self): self.cbs=[]
    def register(self, name, fn): self.cbs.append((name, fn))
class _IP:
    def __init__(self, ns): self.user_ns=ns; self.user_ns_hidden={'_ih'}; self.events=_Events()

def test_bridge_snapshot_uses_ns_and_hidden():
    B.get_ipython = lambda: _IP({'a':1, '_ih':2})
    assert [r.name for r in Bridge().snapshot()]==['a']
def test_on_change_registers_and_fires():
    ip = _IP({}); B.get_ipython = lambda: ip
    hits=[]; ok = on_change(lambda: hits.append(1))
    assert ok is True and ip.events.cbs[0][0]=='post_run_cell'
    ip.events.cbs[0][1](None); assert hits==[1]
def test_no_ipython_is_safe():
    B.get_ipython = lambda: None
    assert Bridge().snapshot()==[] and on_change(lambda: None) is False
test_bridge_snapshot_uses_ns_and_hidden(); test_on_change_registers_and_fires(); test_no_ipython_is_safe()

In [ ]:
def test_bridge_expand():
    B.get_ipython = lambda: _IP({'d': {'x': 1, 'y': 2}})
    kids = Bridge().expand(('d',))
    assert [k.name for k in kids]==["'x'", "'y'"] and kids[0].value=='1'
test_bridge_expand()

In [ ]:
#| export
from paar.snapshot import snapshot, expand
try: from IPython import get_ipython
except Exception:
    def get_ipython(): return None

def _ns():
    ip = get_ipython(); return ip.user_ns if ip else {}

def _hidden():
    ip = get_ipython()
    return frozenset(getattr(ip, 'user_ns_hidden', ())) if ip else frozenset()

class Bridge:
    "Frontend-agnostic access to the live kernel namespace."
    def snapshot(self): return snapshot(_ns(), _hidden())
    def expand(self, accessor): return expand(_ns(), accessor)

def on_change(cb):
    "Register cb() to fire after each cell execution; returns False outside IPython."
    ip = get_ipython()
    if not ip: return False
    ip.events.register('post_run_cell', lambda res: cb())
    return True

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()